# Enable MCP Flag on HTTP Connection

Use this notebook only if the **Is MCP connection** toggle was missing from the UI when you created the connection. Complete Steps 1 and 2 of `MCP-MANUAL-SETUP.md` first; the HTTP connection must already exist.

This notebook reads the connection's existing options (host, port, base path, client ID, scope, token endpoint) and re-sends them with `is_mcp_connection` set to `true`. You only supply the **client secret**, because the API masks it on read and never returns it.

After running all three cells, proceed to `mcp-validate.ipynb` to confirm the connection works.

In [ ]:
# Connection name chosen in Step 1 of MCP-MANUAL-SETUP.md
CONNECTION_NAME = "neo4j_aircraft_mcp_server"

# Client secret from neo4j-agentcore-mcp-server/.mcp-credentials.json (client_secret).
# This is the only value you need to enter: the API masks it on read, so it
# cannot be recovered from the existing connection. Everything else is read back.
CLIENT_SECRET = "<client_secret>"

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# get() returns read-only fields (e.g. access_token_expiration) that update()
# rejects. Keep only the writable options before re-sending.
SETTABLE = {
    "host", "port", "base_path", "client_id", "oauth_scope", "token_endpoint",
    "is_mcp_connection", "client_secret", "payload_logging_destination",
    "on_call_service_policies", "on_call_service_policy_bodies",
    "on_result_service_policies", "on_result_service_policy_bodies",
}

# Read the existing options so we don't have to re-enter host, port, base_path,
# client_id, oauth_scope, and token_endpoint.
conn = w.connections.get(CONNECTION_NAME)
opts = {k: v for k, v in (conn.options or {}).items() if k in SETTABLE}
if not opts:
    raise SystemExit(
        f"No options found for '{CONNECTION_NAME}'. "
        "Confirm the connection exists (Steps 1-2 of MCP-MANUAL-SETUP.md)."
    )

# Overlay only what we must: the MCP flag, and the secret the API can't return.
# update() replaces the full options map, so the read-back values above are
# re-sent unchanged.
opts["is_mcp_connection"] = "true"
opts["client_secret"]     = CLIENT_SECRET

w.connections.update(name=CONNECTION_NAME, options=opts)

print(f"Updated '{CONNECTION_NAME}'.")

In [ ]:
conn   = w.connections.get(CONNECTION_NAME)
opts   = conn.options or {}
is_mcp = opts.get("is_mcp_connection", "false")
url    = conn.url or ""

print(f"is_mcp_connection : {is_mcp}")
print(f"url               : {url}")

if is_mcp == "true" and url.endswith("/mcp"):
    print("\nReady. Proceed to mcp-validate.ipynb.")
else:
    if is_mcp != "true":
        print("\nWARNING: is_mcp_connection is not 'true'. Re-run the update cell.")
    if not url.endswith("/mcp"):
        print(
            "WARNING: url does not end in /mcp. The connection's base_path is wrong "
            "(the UI can default it to '/'). Edit the connection and set Base path to "
            "'/mcp', then re-run this notebook."
        )